# Multilayer Agent Test Runner Notebook

This notebook runs a multilayered test harness for the agent project. It includes environment checks, API test cases, validation rules, pass/fail reporting, metrics, and saved artifacts. Use in-process `TestClient` when possible, and fall back to HTTP if needed.

In [17]:
import sys
import subprocess
from pathlib import Path


ROOT = Path('..').resolve()
NOTEBOOK_ROOT = Path('.').resolve()
OUTPUT_DIR = NOTEBOOK_ROOT / 'agent_test_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def ensure_package(package: str) -> None:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

for pkg in ['fastapi', 'requests', 'pandas', 'numpy', 'pytest', 'httpx']:
    ensure_package(pkg)

import json
import time
import datetime
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
import numpy as np

print('ROOT =', ROOT)
print('Python executable =', sys.executable)
print('Notebook startup complete')


ROOT = C:\nisa_punya\PRESTIJ\data\education_workforce_agent_mvp_en
Python executable = C:\Users\user\anaconda3\python.exe
Notebook startup complete


In [18]:
API_BASE_URL = 'http://127.0.0.1:8002'
USE_TESTCLIENT = False  # Set True only if the notebook environment can import main.app safely
TEST_TIMEOUT_SECONDS = 30

client = None
app = None
if USE_TESTCLIENT:
    try:
        from fastapi.testclient import TestClient
        from main import app
        client = TestClient(app)
    except Exception as exc:
        print('Warning: Cannot load main.app in this notebook environment.')
        print('Falling back to HTTP mode. Start the server separately if you want to use the live API.')
        print('Import error:', exc)
        USE_TESTCLIENT = False

if not USE_TESTCLIENT:
    print('HTTP fallback mode enabled. Make sure the API server is running at', API_BASE_URL)


def get_current_environment() -> Dict[str, Any]:
    return {
        'python_executable': sys.executable,
        'path': str(ROOT),
        'use_testclient': USE_TESTCLIENT,
        'api_base_url': API_BASE_URL,
        'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
    }

env = get_current_environment()
print(json.dumps(env, indent=2))


HTTP fallback mode enabled. Make sure the API server is running at http://127.0.0.1:8002
{
  "python_executable": "C:\\Users\\user\\anaconda3\\python.exe",
  "path": "C:\\nisa_punya\\PRESTIJ\\data\\education_workforce_agent_mvp_en",
  "use_testclient": false,
  "api_base_url": "http://127.0.0.1:8002",
  "timestamp": "2026-06-30T14:47:51.053888Z"
}


C:\Users\user\AppData\Local\Temp\ipykernel_8236\144342277.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',


In [19]:
@dataclass
class TestCaseResult:
    id: str
    category: str
    description: str
    status: str
    duration_seconds: float
    error: Optional[str]
    details: Dict[str, Any]

user_test_cases = [
    {
        'id': 'api_health',
        'category': 'API behavior',
        'description': 'health endpoint returns 200',
        'method': 'get',
        'path': '/api/health',
        'expected_status': 200,
    },
    {
        'id': 'api_filters',
        'category': 'API behavior',
        'description': 'filter endpoints return expected values',
        'method': 'get',
        'path': '/api/filters/kodtingkatantahun',
        'params': {'negeri': 'JOHOR'},
        'expected_status': 200,
        'expected_field': 'values',
    },
    {
        'id': 'api_forecast',
        'category': 'API behavior',
        'description': 'forecast endpoint returns a valid projection',
        'method': 'post',
        'path': '/api/forecast/2027',
        'json': {'subject': 'SAINS', 'negeri': 'JOHOR'},
        'expected_status': 200,
        'expected_field': 'summary',
    },
    {
        'id': 'sim_single_policy',
        'category': 'Simulation logic',
        'description': 'single-policy simulation returns summary',
        'method': 'post',
        'path': '/api/simulate',
        'json': {
            'target_year': 2027,
            'subject': 'SAINS',
            'negeri': 'JOHOR',
            'policy_type': 'option_ratio',
            'option_ratio': 0.70,
        },
        'expected_status': 200,
        'expected_field': 'summary',
    },
    {
        'id': 'sim_combined_policy',
        'category': 'Simulation logic',
        'description': 'combined-policy simulation has policy_impacts',
        'method': 'post',
        'path': '/api/simulate',
        'json': {
            'target_year': 2027,
            'subject': 'SAINS',
            'negeri': 'JOHOR',
            'policy_mode': 'combined',
            'policy_type': 'teaching_hours',
            'active_policies': ['teaching_hours', 'teacher_capacity'],
            'teaching_hours_change_pct': 10,
            'teacher_capacity_change_pct': 5,
        },
        'expected_status': 200,
        'expected_field': 'policy_impacts',
    },
    {
        'id': 'agent_explanation',
        'category': 'Agent orchestration',
        'description': 'explanation text is present',
        'method': 'post',
        'path': '/api/simulate',
        'json': {
            'target_year': 2027,
            'subject': 'SAINS',
            'negeri': 'JOHOR',
            'policy_type': 'option_ratio',
            'option_ratio': 0.70,
        },
        'expected_status': 200,
        'expected_field': 'explanation',
    },
    {
        'id': 'error_invalid_input',
        'category': 'Error handling',
        'description': 'invalid policy_type returns 422',
        'method': 'post',
        'path': '/api/simulate',
        'json': {'target_year': 2027, 'subject': 'SAINS', 'negeri': 'JOHOR', 'policy_type': 'unknown_policy'},
        'expected_status': 422,
    },
    {
        'id': 'error_missing_fields',
        'category': 'Error handling',
        'description': 'missing required fields returns 200 with defaults',
        'method': 'post',
        'path': '/api/simulate',
        'json': {'subject': 'SAINS'},
        'expected_status': 200,
        'expected_field': 'summary',
    },
]

print(f'Loaded {len(user_test_cases)} cases')


Loaded 8 cases


In [20]:
def run_http_case(case: Dict[str, Any]) -> TestCaseResult:
    start = time.time()
    try:
        method = case['method'].lower()
        kwargs: Dict[str, Any] = {}
        if case.get('params') is not None:
            kwargs['params'] = case['params']
        if case.get('json') is not None:
            kwargs['json'] = case['json']
        if USE_TESTCLIENT and client is not None:
            response = client.request(method, case['path'], **kwargs)
        else:
            url = API_BASE_URL + case['path']
            response = requests.request(method, url, timeout=TEST_TIMEOUT_SECONDS, **kwargs)
        duration = time.time() - start
        try:
            content = response.json()
        except Exception:
            content = response.text
        details = {'status_code': response.status_code, 'response': content}
        if response.status_code != case.get('expected_status', 200):
            return TestCaseResult(case['id'], case['category'], case['description'], 'failed', duration, f'Unexpected status {response.status_code}', details)
        expected_field = case.get('expected_field')
        if expected_field and isinstance(content, dict):
            if expected_field not in content:
                return TestCaseResult(case['id'], case['category'], case['description'], 'failed', duration, f'Missing field {expected_field}', details)
        return TestCaseResult(case['id'], case['category'], case['description'], 'passed', duration, None, details)
    except Exception as exc:
        duration = time.time() - start
        return TestCaseResult(case['id'], case['category'], case['description'], 'error', duration, str(exc), {})

results = [run_http_case(case) for case in user_test_cases]
print('Executed', len(results), 'cases')

Executed 8 cases


In [21]:
df = pd.DataFrame([asdict(result) for result in results])
df['duration_seconds'] = df['duration_seconds'].astype(float)
display(df[['id', 'category', 'description', 'status', 'duration_seconds', 'error']])

summary = {
    'total_cases': len(results),
    'passed': sum(1 for result in results if result.status == 'passed'),
    'failed': sum(1 for result in results if result.status != 'passed'),
    'average_latency': float(np.mean([r.duration_seconds for r in results])),
    'max_latency': float(np.max([r.duration_seconds for r in results])),
}
print(json.dumps(summary, indent=2))

,id,category,description,status,duration_seconds,error
0,api_health,API behavior,health endpoint returns 200,passed,1.073193,None
1,api_filters,API behavior,filter endpoints return expected values,passed,0.041860,None
2,api_forecast,API behavior,forecast endpoint returns a valid projection,passed,3.061223,None
3,sim_single_policy,Simulation logic,single-policy simulation returns summary,passed,2.384470,None
4,sim_combined_policy,Simulation logic,combined-policy simulation has policy_impacts,passed,2.445333,None
5,agent_explanation,Agent orchestration,explanation text is present,passed,2.387400,None
6,error_invalid_input,Error handling,invalid policy_type returns 422,passed,0.015219,None
7,error_missing_fields,Error handling,missing required fields returns 200 with defaults,passed,2.195660,None


{
  "total_cases": 8,
  "passed": 8,
  "failed": 0,
  "average_latency": 1.7005448937416077,
  "max_latency": 3.061223268508911
}


In [ ]:
artifact = {
    'environment': env,
    'summary': summary,
    'results': [asdict(result) for result in results],
    'generated_at': datetime.datetime.utcnow().isoformat() + 'Z',
}
artifact_path = OUTPUT_DIR / f'agent_test_runner_multilayer_{datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}.json'
artifact_path.write_text(json.dumps(artifact, indent=2), encoding='utf-8')
print('Saved artifact to', artifact_path)